# Rule-Based EntityRuler Project

**NLP IE — Combine spaCy NER with custom EntityRuler patterns**

## Project Overview

Demonstrate **EntityRuler** placement (before vs after NER) for
domain-specific entity extraction on product reviews.

## Learning Objectives

1. Understand EntityRuler and NER interaction.
2. Add custom patterns before and after NER.
3. Handle pattern priority and overlaps.

## Problem Statement

Most NER applications need domain-specific entities. EntityRuler fills the gap.

## Why This Project Matters

- Custom entity types are always needed.
- Rules are fast and interpretable.
- Combining rules + ML is best practice.

## Dataset Overview

10 synthetic product reviews.

## Dataset Source and License Notes

Synthetic.

## Environment Setup

In [ ]:
import subprocess, sys, importlib
for pkg in ["pandas","numpy","matplotlib","seaborn","spacy"]:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
try:
    import spacy; spacy.load("en_core_web_sm")
except OSError:
    subprocess.check_call([sys.executable,"-m","spacy","download","en_core_web_sm"])
print("Environment ready.")


## Imports

In [ ]:
import re, warnings
from collections import defaultdict
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import spacy
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
%matplotlib inline
nlp = spacy.load("en_core_web_sm")
SEED = 42; np.random.seed(SEED)
print("Loaded.")


## Configuration / Constants

In [ ]:
PRODUCT_PATTERNS = [{"label":"PRODUCT","pattern":p} for p in
    ["iPhone","MacBook","iPad","AirPods","Galaxy S24","Pixel 8","Surface Pro","ThinkPad","PlayStation 5","Xbox Series X","Nintendo Switch"]]
BRAND_PATTERNS = [{"label":"BRAND","pattern":b} for b in
    ["Apple","Samsung","Google","Microsoft","Sony","Lenovo","Nintendo"]]
FEATURE_PATTERNS = [{"label":"FEATURE","pattern":f} for f in
    ["battery life","screen quality","camera","processor","storage","display","keyboard","trackpad","speakers","fast charging"]]
ALL_PATTERNS = PRODUCT_PATTERNS + BRAND_PATTERNS + FEATURE_PATTERNS
print(f"Total patterns: {len(ALL_PATTERNS)}")


## Dataset Download or Loading

In [ ]:
REVIEWS = [
    "The iPhone has incredible camera quality and the battery life lasts all day. Apple really outdid themselves.",
    "Samsung Galaxy S24 has a stunning display and fast charging. The processor is blazing fast.",
    "Microsoft Surface Pro is great for productivity. The keyboard and trackpad are excellent.",
    "I love my MacBook for coding. The screen quality is amazing and the speakers are good.",
    "Google Pixel 8 takes amazing photos. The camera is top notch.",
    "The PlayStation 5 from Sony has incredible graphics.",
    "Nintendo Switch is perfect for family gaming.",
    "Lenovo ThinkPad is the best business laptop. Great keyboard and battery life.",
    "Apple AirPods have seamless integration with iPhone.",
    "Xbox Series X from Microsoft offers great value. The storage is generous.",
]
df = pd.DataFrame({"text": REVIEWS})
print(f"Loaded {len(df)} reviews")


## Data Validation Checks

In [ ]:
assert df["text"].notna().all()
print(f"Validated: {len(df)}")


## Exploratory Data Analysis

In [ ]:
df["words"] = df["text"].str.split().apply(len)
print(f"Avg words: {df["words"].mean():.0f}")


## Target Analysis

Targets: products, brands, features.

## Train/Validation/Test Split Strategy

Not applicable.

## Preprocessing Strategy

Compare EntityRuler **before** vs **after** NER.

## Feature Engineering — Pipeline A: Ruler BEFORE NER

In [ ]:
nlp_before = spacy.load("en_core_web_sm")
ruler_b = nlp_before.add_pipe("entity_ruler", before="ner")
ruler_b.add_patterns(ALL_PATTERNS)
print(f"Pipeline A: {nlp_before.pipe_names}")


## Pipeline B: Ruler AFTER NER

In [ ]:
nlp_after = spacy.load("en_core_web_sm")
ruler_a = nlp_after.add_pipe("entity_ruler", after="ner", config={"overwrite_ents": False})
ruler_a.add_patterns(ALL_PATTERNS)
print(f"Pipeline B: {nlp_after.pipe_names}")


## Baseline Model — Compare Pipelines

In [ ]:
custom_labels = {"PRODUCT","BRAND","FEATURE"}
for name, pipe in [("A (before)", nlp_before), ("B (after)", nlp_after)]:
    total = 0
    print(f"\n--- Pipeline {name} ---")
    for text in REVIEWS[:5]:
        doc = pipe(text)
        custom = [(e.text, e.label_) for e in doc.ents if e.label_ in custom_labels]
        total += len(custom)
        if custom: print(f"  {text[:40]}... -> {custom}")
    print(f"  Total custom entities: {total}")


## LazyPredict Benchmark

> **Not applicable.** NLP IE task.

## FLAML AutoML Run

> **Not applicable.** FLAML not used for NLP IE.

## Additional Best-Library Workflow — Entity Frequency

In [ ]:
counts = defaultdict(lambda: defaultdict(int))
for text in REVIEWS:
    doc = nlp_before(text)
    for e in doc.ents:
        if e.label_ in custom_labels: counts[e.label_][e.text] += 1
fig, axes = plt.subplots(1,3,figsize=(16,4))
for ax, label in zip(axes, ["PRODUCT","BRAND","FEATURE"]):
    c = dict(sorted(counts[label].items(), key=lambda x:-x[1])[:8])
    if c: ax.barh(list(c.keys()), list(c.values()), color="steelblue"); ax.invert_yaxis()
    ax.set_title(f"{label}")
plt.tight_layout(); plt.show()


## Top 3 Model Selection

Pipeline A (before) vs Pipeline B (after).

## Final Training and Evaluation of Top 3

Pipeline A (ruler before NER) is recommended.

## Error Analysis

In [ ]:
for i, t in enumerate(REVIEWS):
    doc = nlp_before(t)
    if not [e for e in doc.ents if e.label_ in custom_labels]:
        print(f"Review {i+1}: no custom entities")


## Interpretation and Business Insight

1. **Before NER** gives custom patterns priority.
2. **After NER** preserves ML, adds non-overlapping.
3. Combining rules + ML is production standard.

## Limitations

- Manual pattern maintenance.
- No fuzzy matching.
- Case-sensitive by default.

## How to Improve This Project

1. Token-pattern matching.
2. PhraseMatcher for large lists.
3. GLiNER for zero-shot.

## Production Considerations

- Version control patterns.
- Use JSONL for storage.
- PhraseMatcher for >10K patterns.

## Common Mistakes

1. Wrong component ordering.
2. Not setting overwrite_ents.
3. Duplicate patterns.

## Mini Challenge / Exercises

1. Add 20 more product patterns.
2. Build token-level patterns.
3. Compare PhraseMatcher speed.

## Final Summary / Key Takeaways

1. EntityRuler adds custom entities.
2. Before NER = custom priority.
3. After NER = ML preserved.
4. Combining rules + ML is standard.